In [1]:
import pandas as pd
import openpyxl
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

## SocioEcon 2050 Summary by Scenario

In [ ]:
base = "../data/processed_data"
out_path = "../SocioEcon_2050_Summary_ByScenario.xlsx"

sum_vars = [
    'total_residential_units', 'total_occ_units',
    'occ_units_low_inc', 'occ_units_med_inc', 'occ_units_high_inc',
    'total_persons',
    'emp_retail', 'emp_srvc', 'emp_rec', 'emp_game', 'emp_other'
]

# Load and sum each scenario
sums = {}
for alt in [1, 2, 3, 4]:
    df = pd.read_csv(f"{base}/Alternative_{alt}_2050/SocioEcon_Summer.csv")
    sums[alt] = {v: int(df[v].sum()) for v in sum_vars}

# Build main data rows
rows = []
for v in sum_vars:
    rows.append({
        'variable': v,
        'scenario_1_sum': sums[1][v],
        'scenario_2_sum': sums[2][v],
        'scenario_3_sum': sums[3][v],
        'scenario_4_sum': sums[4][v],
        'diff_2_vs_1': sums[2][v] - sums[1][v],
        'diff_3_vs_1': sums[3][v] - sums[1][v],
        'diff_4_vs_1': sums[4][v] - sums[1][v],
    })

# Build income % rows
inc_vars = ['occ_units_low_inc', 'occ_units_med_inc', 'occ_units_high_inc']
inc_labels = ['% Low Income', '% Middle Income', '% High Income']
inc_rows = []
for label, v in zip(inc_labels, inc_vars):
    row = {'income_category': label}
    for alt in [1, 2, 3, 4]:
        total_inc = sum(sums[alt][iv] for iv in inc_vars)
        row[f'scenario_{alt}_pct'] = sums[alt][v] / total_inc if total_inc else 0
    inc_rows.append(row)

wb = openpyxl.Workbook()
bold = Font(bold=True)

# ── Sheet 1: Summary ──────────────────────────────────────────────────────────
ws1 = wb.active
ws1.title = 'SocioEcon_2050_Summary'

ws1.append([None, None, None, None, None, 'Absolute Difference', None, None, '% Difference', None, None])
ws1.append(['variable', 'scenario_1_sum', 'scenario_2_sum', 'scenario_3_sum', 'scenario_4_sum',
            'diff_2_vs_1', 'diff_3_vs_1', 'diff_4_vs_1', 2, 3, 4])

for i, row in enumerate(rows, start=3):
    ws1.append([
        row['variable'],
        row['scenario_1_sum'], row['scenario_2_sum'],
        row['scenario_3_sum'], row['scenario_4_sum'],
        row['diff_2_vs_1'], row['diff_3_vs_1'], row['diff_4_vs_1'],
        f'=F{i}/$B{i}', f'=G{i}/$B{i}', f'=H{i}/$B{i}',
    ])

# Format % difference columns
for row in ws1.iter_rows(min_row=3, max_row=2 + len(rows), min_col=9, max_col=11):
    for cell in row:
        cell.number_format = '0.0%'

# Bold headers, merge span headers
for cell in ws1[1]: cell.font = bold
for cell in ws1[2]: cell.font = bold
ws1.merge_cells('F1:H1')
ws1.merge_cells('I1:K1')
ws1['F1'].alignment = Alignment(horizontal='center')
ws1['I1'].alignment = Alignment(horizontal='center')

for col in ws1.columns:
    ws1.column_dimensions[get_column_letter(col[0].column)].width = max(
        (len(str(c.value)) if c.value else 0 for c in col), default=10
    ) + 2

# ── Sheet 2: Income % ─────────────────────────────────────────────────────────
ws2 = wb.create_sheet('Income_Pct_by_Scenario')

ws2.append(['income_category', 'scenario_1', 'scenario_2', 'scenario_3', 'scenario_4'])
for cell in ws2[1]: cell.font = bold

for row in inc_rows:
    ws2.append([
        row['income_category'],
        row['scenario_1_pct'], row['scenario_2_pct'],
        row['scenario_3_pct'], row['scenario_4_pct'],
    ])

for row in ws2.iter_rows(min_row=2, max_row=1 + len(inc_rows), min_col=2, max_col=5):
    for cell in row:
        cell.number_format = '0.0%'

for col in ws2.columns:
    ws2.column_dimensions[get_column_letter(col[0].column)].width = max(
        (len(str(c.value)) if c.value else 0 for c in col), default=14
    ) + 2

wb.save(out_path)
print(f"Saved: {out_path}")

## SocioEcon WFH Summary by Scenario and Year

In [ ]:
emp_cols = ['emp_retail', 'emp_srvc', 'emp_rec', 'emp_game', 'emp_other']

def summarize_socio(path):
    df = pd.read_csv(path)
    total_occ = df['total_occ_units'].sum()
    avg_pph = (
        (df['persons_per_occ_unit'] * df['total_occ_units']).sum() / total_occ
        if total_occ > 0 else 0
    )
    return {
        'total_residential_units':  int(df['total_residential_units'].sum()),
        'total_occ_units':          int(total_occ),
        'occ_units_low_inc':        int(df['occ_units_low_inc'].sum()),
        'occ_units_med_inc':        int(df['occ_units_med_inc'].sum()),
        'occ_units_high_inc':       int(df['occ_units_high_inc'].sum()),
        'avg_persons_per_occ_unit': round(avg_pph, 4),
        'total_persons':            int(df['total_persons'].sum()),
        'total_jobs':               int(df[emp_cols].sum().sum()),
    }

rows = []
for alt in [1, 2, 3, 4]:
    for year in [2035, 2050]:
        for variant, suffix in [('base', ''), ('wfh', '_wfh')]:
            path = f"{base}/Alternative_{alt}_{year}{suffix}/SocioEcon_Summer.csv"
            row = {'scenario': f"Alternative_{alt}", 'year': year, 'variant': variant}
            row.update(summarize_socio(path))
            rows.append(row)

wfh_summary = pd.DataFrame(rows)
wfh_summary.to_csv("wfh_summary.csv", index=False)

In [18]:
emp_cols = ['emp_retail', 'emp_srvc', 'emp_rec', 'emp_game', 'emp_other']

def summarize_socio(path):
    df = pd.read_csv(path)
    total_occ = df['total_occ_units'].sum()
    avg_pph = (
        (df['persons_per_occ_unit'] * df['total_occ_units']).sum() / total_occ
        if total_occ > 0 else 0
    )
    return {
        'total_residential_units':  int(df['total_residential_units'].sum()),
        'total_occ_units':          int(total_occ),
        'occ_units_low_inc':        int(df['occ_units_low_inc'].sum()),
        'occ_units_med_inc':        int(df['occ_units_med_inc'].sum()),
        'occ_units_high_inc':       int(df['occ_units_high_inc'].sum()),
        'avg_persons_per_occ_unit': round(avg_pph, 4),
        'total_persons':            int(df['total_persons'].sum()),
        'total_jobs':               int(df[emp_cols].sum().sum()),
    }

rows = []
for alt in [2, 3, 4]:
    for year in [2035, 2050]:
        for variant, suffix in [('base', ''), ('reduced', '_reduced')]:
            path = f"{base}/Alternative_{alt}{suffix}_{year}/SocioEcon_Summer.csv"
            row = {'scenario': f"Alternative_{alt}", 'year': year, 'variant': variant}
            row.update(summarize_socio(path))
            rows.append(row)

wfh_summary = pd.DataFrame(rows)
wfh_summary.to_csv("reduced_summary.csv", index=False)

In [ ]:
from pathlib import Path

alt_data_base = Path("../Alternative_{alt}/data")

# from each Alternative/data folder import the parquet file as dataframes
parcels = {}
for alt in [1, 2, 3, 4]:
    data_dir = Path(f"../Alternative_{alt}/data")
    parquet_files = sorted(data_dir.glob("sdfParcels_*.parquet"))
    if not parquet_files:
        print(f"No parquet found for Alternative_{alt}")
        continue
    parcels[alt] = pd.read_parquet(parquet_files[-1])
    print(f"Alternative_{alt}: {len(parcels[alt]):,} rows  ({parquet_files[-1].name})")

parcels[1].head()

In [ ]:
# Concatenate with an Alternative label
all_parcels = pd.concat(
    [df.assign(Alternative=f"Alternative_{alt}") for alt, df in parcels.items()],
    ignore_index=True
)

# Total_Residential_Units = existing + forecasted
all_parcels["Total_Residential_Units"] = (
    all_parcels["Residential_Units"].fillna(0)
    + all_parcels["FORECASTED_RESIDENTIAL_UNITS"].fillna(0)
)
# exclude null TAZs
all_parcels = all_parcels.loc[all_parcels["TAZ"].notnull()]
# MF_SF classification based on Residential_Units
all_parcels["MF_SF"] = all_parcels["Total_Residential_Units"].map(
    lambda x: "MF" if x > 1 else ("SF" if x == 1 else "NA")
)

# Group by Alternative and MF_SF, sum Total_Residential_Units
parcel_summary = (
    all_parcels
    .groupby(["Alternative", "MF_SF"], as_index=False)["Total_Residential_Units"]
    .sum()
    .sort_values(["Alternative", "MF_SF"])
    .reset_index(drop=True)
)
parcel_summary

In [ ]:
parcel_summary.to_csv('parcel_summary.csv', index=False)

In [ ]:
parcel = pd.read_parquet(r"C:\Users\amcclary\Documents\GitHub\Transportation\TravelDemandModel\2026_Housing_EIS\Base\data\processed_data\parcel_socioeconomic.parquet")

In [ ]:
parcel['SF_MF'] = parcel['Residential_Units'].map(
    lambda x: "MF" if x > 1 else ("SF" if x == 1 else "NA")
)

# group by sf mf and sum Residential units
parcel_grouped = parcel.groupby("SF_MF", as_index=False)["Residential_Units"].sum()

In [ ]:
# import packages
import pandas as pd
import pathlib
from pathlib import Path
import os
import arcpy
from utils import *
import numpy as np
import pickle
# external connection packages
from sqlalchemy.engine import URL
from sqlalchemy import create_engine

# pandas options
pd.options.mode.copy_on_write = True
pd.options.mode.chained_assignment = None
pd.options.display.max_columns = 999
pd.options.display.max_rows    = 999

# my workspace 
workspace = r"C:\GIS\Scratch.gdb"
# current working directory
local_path = pathlib.Path().absolute()

# get bonus_condit
# set data path as a subfolder of the current working directory TravelDemandModel\2022\
#data_dir = local_path.parents[0] / 'data'
# folder to save processed data
out_dir  = local_path.parents[0] / 'data/processed_data/base_data'
# workspace gdb for stuff that doesnt work in memory
# gdb = os.path.join(local_path,'Workspace.gdb')
gdb = workspace
# set environement workspace to in memory 
arcpy.env.workspace = 'memory'
# # clear memory workspace
# arcpy.management.Delete('memory')

# overwrite true
arcpy.env.overwriteOutput = True
# Set spatial reference to NAD 1983 UTM Zone 10N
sr = arcpy.SpatialReference(26910)

# get parcels from the database
# network path to connection files
filePath = "F:/GIS/PARCELUPDATE/Workspace/"
# database file path 
sdeBase    = os.path.join(filePath, "Vector.sde")
sdeCollect = os.path.join(filePath, "Collection.sde")
sdeTabular = os.path.join(filePath, "Tabular.sde")
sdeEdit    = os.path.join(filePath, "Edit.sde")

# Pickle variables
# part 1 - spatial joins and new categorical fields
parcel_pickle_part1    = out_dir / 'attributed_parcels.geoparquet'


# columsn to list
initial_columns = [ 'APN',
                    'APO_ADDRESS',
                    'Residential_Units',
                    'TouristAccommodation_Units',
                    'CommercialFloorArea_SqFt',
                    'YEAR',
                    'JURISDICTION',
                    'COUNTY',
                    'OWNERSHIP_TYPE',
                    'COUNTY_LANDUSE_DESCRIPTION',
                    'EXISTING_LANDUSE',
                    'REGIONAL_LANDUSE',
                    'YEAR_BUILT',
                    'PLAN_ID',
                    'PLAN_NAME',
                    'ZONING_ID',
                    'ZONING_DESCRIPTION',
                    'TOWN_CENTER',
                    'LOCATION_TO_TOWNCENTER',
                    'TAZ',
                    'PARCEL_ACRES',
                    'PARCEL_SQFT',
                    'WITHIN_BONUSUNIT_BNDY',
                    'WITHIN_TRPA_BNDY',
                    'VHR']

In [ ]:

# read in travel demand model base year parcel data - CHANGE With new name
parcel_base_tdm  = r"Base\data\processed_data\parcel_spatial_joins.parquet"
parcel_base_path = local_path.parents[1].joinpath(parcel_base_tdm)
# read in base year parcel data
sdf_parcel_base  = pd.read_parquet(parcel_base_path)
#output_gdb = r"C:\GIS\Scratch.gdb"
# parcel development layer polygons

# CLEAN THIS UP - We should just populate this in the base year parcel data

# get parcel level data from LTinfo
dfIPES       = pd.read_json("https://www.laketahoeinfo.org/WebServices/GetParcelIPESScores/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476")
dfLCV_LTinfo = pd.read_json('https://www.laketahoeinfo.org/WebServices/GetParcelsByLandCapability/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476')
dfRetired    = pd.read_json("https://www.laketahoeinfo.org/WebServices/GetAllParcels/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476")
dfBankedDev  = pd.read_json('https://www.laketahoeinfo.org/WebServices/GetBankedDevelopmentRights/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476')
dfTransacted = pd.read_json('https://www.laketahoeinfo.org/WebServices/GetTransactedAndBankedDevelopmentRights/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476')
dfAllParcels = pd.read_json('https://www.laketahoeinfo.org/WebServices/GetAllParcels/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476')

# get use tables 
# zoning data
sde_engine = get_conn('sde')
with sde_engine.begin() as conn:
    df_uses    = pd.read_sql("SELECT * FROM sde.SDE.PermissibleUses", conn)
    df_special = pd.read_sql("SELECT * FROM sde.SDE.Special_Designation", conn)
    df_parcel = read_sql_no_geom("SELECT * FROM sde.SDE.Parcel_History_Attributed WHERE YEAR = 2022", conn)

In [ ]:
df_parcel['SF_MF'] = df_parcel['EXISTING_LANDUSE'].map(
    lambda x: "MF" if x in ['Condominium', 'Condominium Common Area','Multi-Family Residential'] else ("SF" if x == 'Single Family Residential' else "NA")
)

df_parcel_2022_grouped = df_parcel.groupby('SF_MF')['Residential_Units'].sum()

In [ ]:
print(df_parcel['EXISTING_LANDUSE'].unique())